# ETL - LAD Batting Dataset


## 1. Imports

In [10]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from dotenv import load_dotenv, find_dotenv
import os





## 2. Extraction
CSV load with statistics

In [11]:
# ── CONFIG ──
CSV_PATH = 'LAD_batting_enriquecido.csv'

# ── LOAD ──
df_raw = pd.read_csv(CSV_PATH)

print(f" Dataset cargado: {df_raw.shape[0]} filas | {df_raw.shape[1]} columnas")
print(f" Rango de años: {df_raw['Year'].min()} – {df_raw['Year'].max()}")
print(f" Jugadores únicos: {df_raw['Name'].nunique()}")
df_raw.head()


 Dataset cargado: 2758 filas | 34 columnas
 Rango de años: 1958 – 2023
 Jugadores únicos: 1040


,Rank,Year,Position,Name,Age,Games,Plate_Appearances,At_Bats,Runs,Hits,...,Double_Plays_Grounded_Into,Times_Hit_By_Pitch,Sacrifice_Hits,Sacrifice_Flies,Intentional_Bases_on_Balls,Dominant_Hand,Switch_Hitter,Birth_Country,Salary,Contract_Type
0,1,2023,C,Will Smith,28,126,554,464,80,121,...,8,15,0,12,2,Right,No,USA,12005000.0,Mid
1,2,2023,1B,Freddie Freeman,33,161,730,637,131,211,...,14,16,0,5,12,Left,No,USA,22813000.0,Star
2,3,2023,2B,Miguel Vargas,23,81,304,256,36,50,...,1,4,1,4,1,Right,No,Cuba,3400000.0,Arbitration
3,4,2023,SS,Miguel Rojas,34,124,423,385,49,91,...,12,5,2,5,0,Right,No,Venezuela,5669000.0,Arbitration
4,5,2023,3B,Max Muncy,32,135,579,482,95,102,...,8,6,0,6,4,Left,No,USA,15065000.0,Star


In [12]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 2758 entries, 0 to 2757
Data columns (total 34 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Rank                                   2758 non-null   int64  
 1   Year                                   2758 non-null   int64  
 2   Position                               2723 non-null   str    
 3   Name                                   2758 non-null   str    
 4   Age                                    2758 non-null   int64  
 5   Games                                  2758 non-null   int64  
 6   Plate_Appearances                      2758 non-null   int64  
 7   At_Bats                                2758 non-null   int64  
 8   Runs                                   2758 non-null   int64  
 9   Hits                                   2758 non-null   int64  
 10  Doubles                                2758 non-null   int64  
 11  Triples        

## 3. Exploration
Review of data 

In [13]:
print("=== TIPOS DE DATOS ===")
print(df_raw.dtypes)
print()
#============================================

print("=== VALORES NULOS ===")
nulls = df_raw.isnull().sum()
print(nulls[nulls > 0])
print()
#============================================

print("=== ESTADÍSTICAS GENERALES ===")
df_raw.describe()


=== TIPOS DE DATOS ===
Rank                                       int64
Year                                       int64
Position                                     str
Name                                         str
Age                                        int64
Games                                      int64
Plate_Appearances                          int64
At_Bats                                    int64
Runs                                       int64
Hits                                       int64
Doubles                                    int64
Triples                                    int64
Home_Runs                                  int64
Runs_Batted_In                             int64
Stolen_Bases                               int64
Caught_Stealing                            int64
Base_On_Balls                              int64
Strikeouts                                 int64
Batting_Average                          float64
On_Base_Percentage                       float

,Rank,Year,Age,Games,Plate_Appearances,At_Bats,Runs,Hits,Doubles,Triples,...,Slugging_Percentage,On_Base_Plus_Slugging_Percentage,On_Base_Plus_Slugging_Percentage_Plus,Total_Bases,Double_Plays_Grounded_Into,Times_Hit_By_Pitch,Sacrifice_Hits,Sacrifice_Flies,Intentional_Bases_on_Balls,Salary
count,2758.000000,2758.000000,2758.000000,2758.000000,2758.000000,2758.000000,2758.000000,2758.000000,2758.000000,2758.000000,...,2758.000000,2758.000000,2758.00000,2758.000000,2758.000000,2758.000000,2758.000000,2758.000000,2758.000000,2.758000e+03
mean,21.974257,1993.071791,28.278463,51.371284,144.608412,128.435460,16.302030,32.963017,5.521030,0.739304,...,0.243686,0.461131,40.52393,50.123278,2.734953,0.940899,1.798042,1.017041,1.266860,1.255308e+06
std,13.084352,19.154014,4.616844,47.071118,199.888339,177.924748,25.880879,49.921789,8.909008,1.656979,...,0.202536,0.351285,83.76642,78.149096,4.359242,2.024458,3.070949,1.898418,2.800694,3.863358e+06
min,1.000000,1958.000000,17.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,-100.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.000000e+03
25%,11.000000,1977.000000,25.000000,13.000000,4.000000,4.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.075500,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.600000e+04
50%,21.000000,1995.000000,28.000000,35.000000,47.000000,42.000000,3.000000,7.000000,1.000000,0.000000,...,0.264500,0.526500,46.00000,9.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.395000e+05
75%,32.000000,2010.000000,31.000000,79.000000,214.000000,191.000000,22.000000,45.000000,8.000000,1.000000,...,0.383000,0.708000,96.00000,69.000000,4.000000,1.000000,2.000000,1.000000,1.000000,6.270000e+05
max,61.000000,2023.000000,49.000000,165.000000,759.000000,695.000000,131.000000,230.000000,59.000000,16.000000,...,2.000000,3.000000,774.00000,376.000000,27.000000,19.000000,25.000000,13.000000,25.000000,3.500000e+07


## 4. Quality assurance



In [14]:
errores = []

# Rule 1: Batting Average must be between 0 y 1
inv_avg = df_raw[df_raw['Batting_Average'] > 1]
if not inv_avg.empty:
    errores.append(f" {len(inv_avg)} filas con Batting_Average > 1")
else:
    print(" Batting_Average: valid values (0–1)")
#=========================================================================

# Rule 2:  OPS can not be negative
inv_ops = df_raw[df_raw['On_Base_Plus_Slugging_Percentage'] < 0]
if not inv_ops.empty:
    errores.append(f" {len(inv_ops)} filas con OPS negativo")
else:
    print(" OPS: valid values")
#=========================================================================

# Rule 3:  Salary can't be negative or cero
inv_sal = df_raw[df_raw['Salary'] <= 0]
if not inv_sal.empty:
    errores.append(f" {len(inv_sal)} filas con Salary <= 0")
else:
    print("salary clean")
#=========================================================================

# Rule 4: Name cant be null
inv_name = df_raw[df_raw['Name'].isnull()]
if not inv_name.empty:
    errores.append(f" {len(inv_name)} filas sin nombre de jugador")
else:
    print(" Name: valid values")



 Batting_Average: valid values (0–1)
 OPS: valid values
salary clean
 Name: valid values


## 5. Transform
Data transform, null data type manage


In [15]:
df = df_raw.copy()

# data type corrections 
df['Year'] = df['Year'].astype(int)
df['Age'] = df['Age'].astype(int)
df['Salary'] = df['Salary'].astype(float)
#=====================================================================

#  columns standarizations 
df['Name']          = df['Name'].str.strip().str.title()
df['Position']      = df['Position'].str.strip().str.upper()
df['Dominant_Hand'] = df['Dominant_Hand'].str.strip().str.title()
df['Switch_Hitter'] = df['Switch_Hitter'].str.strip().str.title()
df['Birth_Country'] = df['Birth_Country'].str.strip().str.title()
df['Contract_Type'] = df['Contract_Type'].str.strip().str.title()
#=====================================================================


#  column [position] null data management 
print(f"nulls before: {df['Position'].isnull().sum()}")
df['Position'] = df['Position'].fillna('Unknown')
print(f"nulls after: {df['Position'].isnull().sum()}")
#==========================================================================


#  delete duplicates 
dupes_before = df.duplicated().sum()
df = df.drop_duplicates()
print(f"duplicates deleted: {dupes_before}")
#============================================================================


#  new column
def assign_era(year):
    if year < 1970:
        return 'Expansion Era'
    elif year < 1980:
        return 'Free Agency Era'
    elif year < 1994:
        return 'Competitive Era'
    elif year < 2006:
        return 'Steroid Era'
    elif year < 2016:
        return 'Post-Steroid Era'
    else:
        return 'Analytics Era'

df['Era'] = df['Year'].apply(assign_era)
#================================================================================


# new column 
def age_category(age):
    if age < 25:
        return 'Young'
    elif age <= 29:
        return 'Peak'
    elif age <= 33:
        return 'Veteran'
    else:
        return 'Senior'

df['Age_Category'] = df['Age'].apply(age_category)

print(f"\n complete transformation: {df.shape[0]} filas | {df.shape[1]} columnas")
print(f"   asigned eras: {df['Era'].value_counts().to_dict()}")
print(f"   age categories: {df['Age_Category'].value_counts().to_dict()}")


nulls before: 35
nulls after: 0
duplicates deleted: 0

 complete transformation: 2758 filas | 36 columnas
   asigned eras: {'Competitive Era': 550, 'Steroid Era': 523, 'Post-Steroid Era': 488, 'Expansion Era': 425, 'Analytics Era': 413, 'Free Agency Era': 359}
   age categories: {'Peak': 1080, 'Veteran': 652, 'Young': 632, 'Senior': 394}


## 6. Normalization
table creation
**structure:**
- dim_jugador — player info
- dim_posicion — pisition
- dim_temporada — ages
- fact_battin — statistics
- fact_contrato — salary, contract types


In [16]:
# ── DIM_JUGADOR ──
dim_jugador = df[['Name', 'Dominant_Hand', 'Switch_Hitter', 'Birth_Country']].drop_duplicates(subset='Name').reset_index(drop=True)
dim_jugador.insert(0, 'player_id', range(1, len(dim_jugador) + 1))
dim_jugador.columns = ['player_id', 'name', 'dominant_hand', 'switch_hitter', 'birth_country']
print(f" dim_jugador: {dim_jugador.shape[0]} unique players")
#=========================================================================================================================================

# ── DIM_POSICION ──
dim_posicion = df[['Position']].drop_duplicates().reset_index(drop=True)
dim_posicion.insert(0, 'position_id', range(1, len(dim_posicion) + 1))
dim_posicion.columns = ['position_id', 'position_code']

position_desc = {
    'C': 'Catcher', '1B': 'First Base', '2B': 'Second Base',
    'SS': 'Shortstop', '3B': 'Third Base', 'LF': 'Left Field',
    'CF': 'Center Field', 'RF': 'Right Field', 'DH': 'Designated Hitter',
    'UT': 'Utility', 'OF': 'Outfield', 'IF': 'Infield',
    'P': 'Pitcher', 'MI': 'Middle Infield', 'CI': 'Corner Infield',
    'UNKNOWN': 'Unknown'
}
dim_posicion['position_name'] = dim_posicion['position_code'].map(position_desc).fillna('Other')

position_cat = {
    'C': 'Battery', '1B': 'Infield', '2B': 'Infield', 'SS': 'Infield',
    '3B': 'Infield', 'LF': 'Outfield', 'CF': 'Outfield', 'RF': 'Outfield',
    'DH': 'Designated', 'UT': 'Utility', 'OF': 'Outfield', 'IF': 'Infield',
    'P': 'Battery', 'MI': 'Infield', 'CI': 'Infield', 'UNKNOWN': 'Unknown'
}
dim_posicion['position_category'] = dim_posicion['position_code'].map(position_cat).fillna('Other')
print(f" dim_posicion: {dim_posicion.shape[0]} positions")
#=========================================================================================================================================


# ── DIM_TEMPORADA ──
dim_temporada = df[['Year', 'Era']].drop_duplicates(subset='Year').sort_values('Year').reset_index(drop=True)
dim_temporada.insert(0, 'season_id', range(1, len(dim_temporada) + 1))
dim_temporada.columns = ['season_id', 'year', 'era']
print(f" dim_temporada: {dim_temporada.shape[0]} temps ({dim_temporada['year'].min()}–{dim_temporada['year'].max()})")
#=========================================================================================================================================


# ── Merge to obtain de id of the principal dataset
df_fact = df.merge(dim_jugador[['player_id', 'name']], left_on='Name', right_on='name', how='left')
df_fact = df_fact.merge(dim_posicion[['position_id', 'position_code']], left_on='Position', right_on='position_code', how='left')
df_fact = df_fact.merge(dim_temporada[['season_id', 'year']], left_on='Year', right_on='year', how='left')

#=========================================================================================================================================

# ── FACT_BATTING ──
fact_batting = df_fact[[
    'player_id', 'position_id', 'season_id', 'Age', 'Age_Category',
    'Games', 'Plate_Appearances', 'At_Bats', 'Runs', 'Hits',
    'Doubles', 'Triples', 'Home_Runs', 'Runs_Batted_In',
    'Stolen_Bases', 'Caught_Stealing', 'Base_On_Balls', 'Strikeouts',
    'Batting_Average', 'On_Base_Percentage', 'Slugging_Percentage',
    'On_Base_Plus_Slugging_Percentage', 'On_Base_Plus_Slugging_Percentage_Plus',
    'Total_Bases', 'Times_Hit_By_Pitch', 'Sacrifice_Hits', 'Sacrifice_Flies',
    'Intentional_Bases_on_Balls', 'Double_Plays_Grounded_Into'
]].copy()
fact_batting.columns = [
    'player_id', 'position_id', 'season_id', 'age', 'age_category',
    'games', 'plate_appearances', 'at_bats', 'runs', 'hits',
    'doubles', 'triples', 'home_runs', 'runs_batted_in',
    'stolen_bases', 'caught_stealing', 'base_on_balls', 'strikeouts',
    'batting_average', 'on_base_percentage', 'slugging_percentage',
    'ops', 'ops_plus', 'total_bases', 'times_hit_by_pitch',
    'sacrifice_hits', 'sacrifice_flies', 'intentional_walks', 'double_plays'
]
fact_batting.insert(0, 'batting_id', range(1, len(fact_batting) + 1))
print(f" fact_batting: {fact_batting.shape[0]} records")
#=========================================================================================================================================


# ── FACT_CONTRATO ──
fact_contrato = df_fact[['player_id', 'season_id', 'Salary', 'Contract_Type']].copy()
fact_contrato.columns = ['player_id', 'season_id', 'salary', 'contract_type']
fact_contrato.insert(0, 'contract_id', range(1, len(fact_contrato) + 1))
print(f" fact_contrato: {fact_contrato.shape[0]} records")

print("\n completed normalization — 3 dimensions + 2 facts table")


 dim_jugador: 1039 unique players
 dim_posicion: 16 positions
 dim_temporada: 66 temps (1958–2023)
 fact_batting: 2758 records
 fact_contrato: 2758 records

 completed normalization — 3 dimensions + 2 facts table


## 7. tables preview

In [17]:
print(" dim_jugador ")
display(dim_jugador.head())

print("\n dim_posicion")
display(dim_posicion.head())

print("\n dim_temporada ")
display(dim_temporada.head())

print("\n fact_batting ")
display(fact_batting.head())

print("\n fact_contrato ")
display(fact_contrato.head())


 dim_jugador 


,player_id,name,dominant_hand,switch_hitter,birth_country
0,1,Will Smith,Right,No,Usa
1,2,Freddie Freeman,Left,No,Usa
2,3,Miguel Vargas,Right,No,Cuba
3,4,Miguel Rojas,Right,No,Venezuela
4,5,Max Muncy,Left,No,Usa



 dim_posicion


,position_id,position_code,position_name,position_category
0,1,C,Catcher,Battery
1,2,1B,First Base,Infield
2,3,2B,Second Base,Infield
3,4,SS,Shortstop,Infield
4,5,3B,Third Base,Infield



 dim_temporada 


,season_id,year,era
0,1,1958,Expansion Era
1,2,1959,Expansion Era
2,3,1960,Expansion Era
3,4,1961,Expansion Era
4,5,1962,Expansion Era



 fact_batting 


,batting_id,player_id,position_id,season_id,age,age_category,games,plate_appearances,at_bats,runs,...,on_base_percentage,slugging_percentage,ops,ops_plus,total_bases,times_hit_by_pitch,sacrifice_hits,sacrifice_flies,intentional_walks,double_plays
0,1,1,1,66,28,Peak,126,554,464,80,...,0.359,0.438,0.797,114,203,15,0,12,2,8
1,2,2,2,66,33,Veteran,161,730,637,131,...,0.410,0.567,0.976,161,361,16,0,5,12,14
2,3,3,3,66,23,Young,81,304,256,36,...,0.305,0.367,0.672,81,94,4,1,4,1,1
3,4,4,4,66,34,Senior,124,423,385,49,...,0.290,0.322,0.612,66,124,5,2,5,0,12
4,5,5,5,66,32,Veteran,135,579,482,95,...,0.333,0.475,0.808,115,229,6,0,6,4,8



 fact_contrato 


,contract_id,player_id,season_id,salary,contract_type
0,1,1,66,12005000.0,Mid
1,2,2,66,22813000.0,Star
2,3,3,66,3400000.0,Arbitration
3,4,4,66,5669000.0,Arbitration
4,5,5,66,15065000.0,Star


## 8. Load to postgres
conection to the db and tables load


In [18]:
tablaz = '''
CREATE TABLE dim_jugador (
    player_id     INT PRIMARY KEY,
    name          VARCHAR(100) NOT NULL UNIQUE,
    dominant_hand VARCHAR(50) NOT NULL,
    switch_hitter VARCHAR(50) NOT NULL,
    birth_country VARCHAR(50) NOT NULL
);

CREATE TABLE dim_posicion (
    position_id       INT PRIMARY KEY,
    position_code     VARCHAR(50) NOT NULL,
    position_name     VARCHAR(50) NOT NULL,
    position_category VARCHAR(50) NOT NULL
);

CREATE TABLE dim_temporada (
    season_id INT PRIMARY KEY,
    year      INT NOT NULL,
    era       VARCHAR(50) NOT NULL
);

CREATE TABLE fact_batting (
    batting_id          INT PRIMARY KEY,
    player_id           INT NOT NULL,
    position_id         INT NOT NULL,
    season_id           INT NOT NULL,
    age                 INT NOT NULL,
    age_category        VARCHAR(50) NOT NULL,
    games               INT NOT NULL,
    plate_appearances   INT NOT NULL,
    at_bats             INT NOT NULL,
    runs                INT NOT NULL,
    hits                INT NOT NULL,
    doubles             INT NOT NULL,
    triples             INT NOT NULL,
    home_runs           INT NOT NULL,
    runs_batted_in      INT NOT NULL,
    stolen_bases        INT NOT NULL,
    caught_stealing     INT NOT NULL,
    base_on_balls       INT NOT NULL,
    strikeouts          INT NOT NULL,
    batting_average     FLOAT NOT NULL,
    on_base_percentage  FLOAT NOT NULL,
    slugging_percentage FLOAT NOT NULL,
    ops                 FLOAT NOT NULL,
    ops_plus            INT NOT NULL,
    total_bases         INT NOT NULL,
    times_hit_by_pitch  INT NOT NULL,
    sacrifice_hits      INT NOT NULL,
    sacrifice_flies     INT NOT NULL,
    intentional_walks   INT NOT NULL,
    double_plays        INT NOT NULL,
    FOREIGN KEY (player_id)   REFERENCES dim_jugador(player_id),
    FOREIGN KEY (position_id) REFERENCES dim_posicion(position_id),
    FOREIGN KEY (season_id)   REFERENCES dim_temporada(season_id)
);

CREATE TABLE fact_contrato (
    contract_id   INT PRIMARY KEY,
    player_id     INT NOT NULL,
    season_id     INT NOT NULL,
    salary        FLOAT NOT NULL,
    contract_type VARCHAR(50) NOT NULL,
    FOREIGN KEY (player_id) REFERENCES dim_jugador(player_id),
    FOREIGN KEY (season_id) REFERENCES dim_temporada(season_id)
);
'''


load_dotenv(find_dotenv())


def get_engine(): 
    usuario = os.getenv("DB_USER", "postgres")
    password = os.getenv("DB_PASSWORD", "0000")
    host = os.getenv("DB_HOST", "localhost")
    puerto = os.getenv("DB_PORT", "5432")
    database = os.getenv("DB_DATABASE", "lad_baseball")

    url = (
            f"postgresql+psycopg2://{usuario}:{password}@{host}:{puerto}/{database}?client_encoding=utf8"
        )

    return create_engine(url)

engine = get_engine()

#======================================================
#table create 
with engine.connect() as con:
    con.execute(text(tablaz))
    con.commit()
    print("tables created succesfully.")

tablas = [
    ('dim_jugador',   dim_jugador),
    ('dim_posicion',  dim_posicion),
    ('dim_temporada', dim_temporada),
    ('fact_batting',  fact_batting),
    ('fact_contrato', fact_contrato),
]

for nombre, tabla in tablas:
    tabla.to_sql(
        name      = nombre,
        con       = engine,
        if_exists = 'append',
        index     = False
    )
    print(f" {nombre}: {len(tabla)} loaded rows")



python-dotenv could not parse statement starting at line 1


tables created succesfully.
 dim_jugador: 1039 loaded rows
 dim_posicion: 16 loaded rows
 dim_temporada: 66 loaded rows
 fact_batting: 2758 loaded rows
 fact_contrato: 2758 loaded rows


## 9. verification

In [19]:
with engine.connect() as conn:
    for nombre, _ in tablas:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {nombre}"))
        count = result.scalar()
        print(f" {nombre}: {count} rows loaded on postgre")


 dim_jugador: 1039 rows loaded on postgre
 dim_posicion: 16 rows loaded on postgre
 dim_temporada: 66 rows loaded on postgre
 fact_batting: 2758 rows loaded on postgre
 fact_contrato: 2758 rows loaded on postgre
